# Test the trained summarizer

Loads the model produced by `01_train_bart_cnn_dailymail.ipynb` and summarizes custom texts (news, a research abstract and a long research-paper-style document) with the abstractive model, TextRank and LEAD-3 side by side.

**Where the model is found (automatically):**
* **Kaggle:** create a new notebook → *Add Input* → *Notebook Output Files* → select your training notebook. The model folder, `src/` and `test_cases/` are all in its output.
* **Local / Colab:** clone the repo, unzip `bart-base-cnn-summarizer.zip` into `models/`, open this notebook from `notebooks/`.
* Or set the environment variable `SUMM_MODEL_DIR` / the variable `MODEL_DIR` below.

CPU is fine for a few texts (≈5–20 s per summary); a GPU is faster.

In [ ]:
import os, re, sys, glob, json, time, zipfile, subprocess

MODEL_DIR = os.environ.get("SUMM_MODEL_DIR")   # None -> auto-detect
FALLBACK_MODEL = None                          # e.g. "facebook/bart-large-cnn" to try the notebook before your model exists
REPO_URL = "https://github.com/ksidharth8/text-summarization-transformers"
SEARCH_ROOTS = ["/kaggle/input", "/kaggle/working", "models", "../models", "outputs", "../outputs"]


def find_model_dir():
    for r in SEARCH_ROOTS:
        hits = sorted(glob.glob(os.path.join(r, "**", "summarizer_config.json"), recursive=True))
        if hits:
            return os.path.dirname(os.path.abspath(hits[0]))
    for r in SEARCH_ROOTS:                                    # only the zip is there -> extract it
        for z in sorted(glob.glob(os.path.join(r, "**", "*-cnn-summarizer.zip"), recursive=True)):
            dest = os.path.abspath("models")
            os.makedirs(dest, exist_ok=True)
            with zipfile.ZipFile(z) as zf:
                zf.extractall(dest)
            hits = glob.glob(os.path.join(dest, "**", "summarizer_config.json"), recursive=True)
            if hits:
                return os.path.dirname(hits[0])
    return None


def find_src_root(model_dir):
    cands = [os.environ.get("SUMM_SRC_ROOT"), "..", ".", os.path.dirname(model_dir) if model_dir else None, "/kaggle/working"]
    cands += [os.path.dirname(os.path.dirname(p)) for p in glob.glob("/kaggle/input/**/src/summarizer.py", recursive=True)]
    for c in cands:
        if c and os.path.isfile(os.path.join(c, "src", "summarizer.py")):
            return os.path.abspath(c)
    if not os.path.isdir("text-summarization-transformers"):
        subprocess.run(["git", "clone", "-q", REPO_URL], check=False)
    if os.path.isfile("text-summarization-transformers/src/summarizer.py"):
        return os.path.abspath("text-summarization-transformers")
    raise FileNotFoundError("src/ package not found: run this notebook from the repo's notebooks/ folder, "
                            "or attach the training notebook's output on Kaggle.")


MODEL_DIR = MODEL_DIR or find_model_dir() or FALLBACK_MODEL
if MODEL_DIR is None:
    raise FileNotFoundError("No trained model found. Attach the training notebook output (Kaggle) or unzip "
                            "bart-base-cnn-summarizer.zip into models/ - or set FALLBACK_MODEL.")
SRC_ROOT = find_src_root(MODEL_DIR if os.path.isdir(str(MODEL_DIR)) else None)
sys.path.insert(0, SRC_ROOT)
print("model:", MODEL_DIR, "\nsrc  :", SRC_ROOT)

In [ ]:
import torch
import transformers
from IPython.display import Markdown, display
transformers.logging.set_verbosity_error()

from src.summarizer import Summarizer
from src.extractive import textrank, lead_k
from src.metrics import novel_ngram_ratio

summarizer = Summarizer(MODEL_DIR)
print(f"{summarizer.num_parameters / 1e6:.1f} M parameters on {summarizer.device} | max input {summarizer.max_source_len} tokens")
print("decoding:", summarizer.generation)
if os.path.isfile(os.path.join(str(MODEL_DIR), "summarizer_config.json")):
    meta = json.load(open(os.path.join(MODEL_DIR, "summarizer_config.json")))
    print("trained:", {k: meta.get(k) for k in ("base_model", "run_mode", "steps", "samples_seen", "train_hours", "created")})

In [ ]:
TEST_CASES = {
    '01_news_electric_buses': """The municipal corporation of a mid-sized hill city has approved a plan to replace its entire diesel bus fleet with 120 electric buses by the end of 2028, officials said on Monday. The first batch of 30 buses is expected to arrive in March and will run on the four busiest routes, which together carry about 65,000 passengers a day.

The project will cost an estimated 210 crore rupees, with roughly half the money coming from a national clean-transport grant and the rest from a low-interest loan. Two new depots with fast chargers will be built on the northern and southern edges of the city, each able to charge 40 buses overnight.

Transport officials said the switch would cut the corporation's fuel bill by nearly 40 per cent and remove around 9,000 tonnes of carbon dioxide emissions every year. Ticket prices will stay the same for at least two years, they added.

Not everyone is convinced. A local commuters' association welcomed the cleaner buses but warned that steep hill roads could drain batteries faster than expected, especially during the monsoon, when air-conditioning use rises. The association asked the corporation to publish range data from a three-month trial before placing the full order.

The corporation said a trial with two leased buses had already covered 18,000 kilometres without a single breakdown, and that the tender requires manufacturers to guarantee a range of at least 220 kilometres on a single charge under full load.""",
    '02_news_flood_warning_system': """Villages along a flood-prone river basin in eastern India will receive flood warnings by SMS and voice call up to 36 hours in advance under a new early-warning system launched by the state disaster management authority on Thursday.

The system combines rainfall readings from 140 automatic weather stations with water-level sensors installed on 26 bridges and embankments. A hydrological model updates forecasts every hour and sends alerts in Hindi, English and two tribal languages to registered mobile numbers in the affected blocks. Villagers without phones will be warned through loudspeakers mounted on panchayat buildings.

Officials said the pilot, which ran in three districts during last year's monsoon, sent 41 alerts and correctly predicted 37 of the 39 floods that occurred, with an average lead time of 22 hours. Two alerts turned out to be false alarms. During the pilot, the number of cattle lost to flooding in the three districts fell by more than half compared with the previous five-year average.

The authority plans to extend the network to all 24 districts of the state within two years at a cost of about 85 crore rupees. It is also training 3,000 volunteers, most of them women from self-help groups, to guide evacuations and maintain the sensors.

Experts welcomed the move but cautioned that warnings are only useful if people trust them. A hydrologist who reviewed the pilot said the authority should publish its accuracy figures every season and explain every false alarm publicly, so that villagers do not start ignoring the messages.""",
    '03_research_abstract': """Large language models are increasingly deployed to summarize scientific articles, yet their summaries frequently contain statements that are not supported by the source document. In this work we study how the length of the input context affects the factual consistency of abstractive summaries of biomedical papers. We construct a benchmark of 2,400 open-access articles paired with expert-written lay summaries and annotate 9,600 model-generated sentences for unsupported claims. We compare three strategies for handling documents that exceed the model's context window: truncation to the first 1,024 tokens, retrieval of the most relevant sections, and hierarchical summarization in which each section is summarized independently before a final summary is produced. Hierarchical summarization improves ROUGE-L by 2.1 points over truncation and reduces the rate of unsupported sentences from 18.4 per cent to 11.7 per cent, whereas retrieval gives smaller gains on both measures. Most remaining errors involve numerical values and the direction of reported effects, which suggests that numeric grounding is a promising target for future work. We release the benchmark, the annotations and our evaluation code to support research on faithful summarization of long scientific documents.""",
    '04_long_paper_token_pruning': """Adaptive Token Pruning for Efficient Transformer Inference on Edge Devices

Abstract. Transformer models achieve strong accuracy on language and vision tasks, but their inference cost grows quadratically with the number of input tokens, which makes deployment on phones and embedded boards difficult. We propose an adaptive token pruning method that removes uninformative tokens layer by layer, using a lightweight scoring head trained jointly with the model. On three text classification benchmarks and one image classification benchmark, the method reduces inference latency by 38 to 52 per cent while losing less than one percentage point of accuracy.

1. Introduction. Edge devices such as smartphones, drones and industrial sensors increasingly run neural networks locally in order to protect privacy, reduce network traffic and respond in real time. Transformers are attractive for these settings because a single architecture can handle text, images and audio. However, self-attention compares every token with every other token, so the computation and memory required grow with the square of the sequence length. Many tokens contribute little to the final prediction. In a sentence classification task, for example, punctuation, articles and repeated words can often be ignored after the first few layers without changing the output. Existing pruning methods usually remove a fixed fraction of tokens at predefined layers. Such fixed schedules are simple, but they waste computation on easy inputs and prune too aggressively on difficult ones. Our goal is a pruning policy that adapts to each input and to the latency budget of the target device.

2. Method. We attach a small scoring head, consisting of a single linear layer followed by a sigmoid, to selected transformer layers. The head assigns each token a keep probability based on its hidden state. During training, tokens are sampled with a straight-through estimator so that the scoring head receives gradients from the task loss. We add a budget loss that penalizes the expected number of floating-point operations whenever it exceeds a target value chosen by the user. At inference time, tokens whose keep probability falls below a threshold are removed, and the remaining tokens are packed into a shorter sequence, which reduces the cost of all following layers. The classification token is never pruned. Because the threshold can be changed after training, a single trained model can serve devices with different latency budgets.

3. Experimental setup. We evaluate the method on three text benchmarks covering sentiment analysis, topic classification and natural language inference, and on one image benchmark with one thousand classes. For text we fine-tune a twelve-layer encoder with 110 million parameters; for images we use a vision transformer of similar size that splits each image into 196 patches. Latency is measured on a mid-range smartphone processor and on a single-board computer with an embedded graphics unit, using batch size one, which reflects typical on-device usage. We compare against the unpruned model, two fixed-schedule pruning methods and a distilled six-layer model.

4. Results. On the text benchmarks, adaptive pruning reduces average latency by 38 per cent on the smartphone and 44 per cent on the single-board computer, with an average accuracy drop of 0.6 percentage points. Fixed-schedule pruning achieves similar speed-ups only at the cost of accuracy drops between 1.8 and 3.1 points. The distilled model is faster than all pruning variants, but it loses 2.4 points of accuracy on natural language inference, the most difficult task. On the image benchmark, adaptive pruning removes on average 61 per cent of the patches by the final layer and reduces latency by 52 per cent with a top-1 accuracy drop of 0.9 points. Visualizations show that the retained patches concentrate on the object, while background regions such as sky and walls are removed early.

5. Analysis. The number of retained tokens varies widely between inputs. Short, unambiguous reviews keep fewer than a quarter of their tokens after the sixth layer, whereas long reviews with negation or sarcasm keep more than half. This behaviour explains why adaptive pruning preserves accuracy better than fixed schedules. An ablation study shows that the budget loss is essential: without it, the scoring head learns to keep almost every token, and the speed-up disappears. Scoring heads placed only in the middle layers give the best trade-off, because early layers have not yet built useful representations and late layers leave little computation to save.

6. Limitations. Our experiments use encoder-only models and classification tasks. Generative tasks such as summarization and translation require the decoder to attend to pruned positions, which our method does not yet support. In addition, latency gains depend on the efficiency of the sequence-packing operation, which is well optimized on the processors we tested but may be slower on other hardware.

7. Conclusion. Adaptive token pruning offers a simple way to make transformer inference cheaper on edge devices while keeping accuracy close to that of the full model. Because the pruning threshold can be adjusted after training, one model can be deployed across devices with different performance limits. Future work will extend the approach to encoder-decoder models and combine it with weight quantization.""",
}

# prefer the files in the repository / training output if present (so you can add your own .txt files)
_files = sorted(glob.glob(os.path.join(SRC_ROOT, "test_cases", "*.txt")))
if _files:
    TEST_CASES = {os.path.splitext(os.path.basename(p))[0]: open(p, encoding="utf-8").read().strip() for p in _files}
print(f"{len(TEST_CASES)} test cases:", list(TEST_CASES))

In [ ]:
def compare(text, name="input"):
    n_tok = summarizer.count_tokens(summarizer.prepare(text))
    budget = summarizer.max_source_len - summarizer.count_tokens(summarizer.prefix) - 8
    t0 = time.time()
    if n_tok > budget:
        out = summarizer.summarize_long(text)
        abstractive, mode = out["summary"], f"long-document map-reduce ({out['chunks']} chunks, {out['rounds']} round(s))"
    else:
        out, abstractive, mode = None, summarizer.summarize(text), "single pass"
    secs = time.time() - t0
    nov = novel_ngram_ratio([text], [abstractive], ns=(1, 2))
    display(Markdown(
        f"### {name}\n*{len(text.split())} words / {n_tok} tokens — {mode} — {secs:.1f}s — novel bigrams {nov['2']}%*\n\n"
        f"**Abstractive:** {abstractive}\n\n**TextRank:** {textrank(text, 3).replace(chr(10), ' ')}\n\n"
        f"**LEAD-3:** {lead_k(text, 3).replace(chr(10), ' ')}\n"))
    return out


LONG_RESULTS = {}
for name, text in TEST_CASES.items():
    out = compare(text, name)
    if out:
        LONG_RESULTS[name] = out

In [ ]:
# Long-document mode in detail: chunk summaries (map) -> final summary (reduce)
for name, out in LONG_RESULTS.items():
    print(f"{name}: {out['input_tokens']} tokens -> {out['chunks']} chunks, {out['rounds']} round(s)")
    for r, partials in enumerate(out["partial_summaries"], 1):
        print(f"\n  round {r}:")
        for j, p in enumerate(partials, 1):
            print(f"   [{j}] {p}")
    print("\n  FINAL:", out["summary"])

In [ ]:
# Try your own text -------------------------------------------------------------------------------
MY_TEXT = """Paste any news article, report or paper section here. Longer texts are summarized in several
chunks automatically. The summary length adapts to the input length for short texts."""
_ = compare(MY_TEXT, "my text")

# Decoding can be changed per call, e.g. shorter and more focused:
# summarizer.summarize(MY_TEXT, num_beams=6, max_new_tokens=60, min_new_tokens=20, length_penalty=1.0)

In [ ]:
# Optional: quick ROUGE check on a few CNN/DailyMail test articles (if the dataset is attached)
EVAL_SAMPLES = 100
_root = os.environ.get("SUMM_DATA_DIR", "/kaggle/input")
_csvs = [p for p in glob.glob(os.path.join(_root, "**", "*.csv"), recursive=True) if "test" in os.path.basename(p).lower()]
_csvs = [p for p in _csvs if re.search(r"cnn|dailymail", p, re.I)] or _csvs
try:
    import re
    import pandas as pd
    from src.metrics import evaluate_system
    from src.preprocessing import clean_summary
    df = pd.read_csv(_csvs[0]).dropna(subset=["article", "highlights"])
    df = df.sample(n=min(EVAL_SAMPLES, len(df)), random_state=0)
    refs = [clean_summary(h) for h in df["highlights"]]
    preds = summarizer.summarize(df["article"].tolist(), batch_size=16, adaptive_length=False)
    res, _ = evaluate_system(preds, refs)
    print(f"{len(df)} random test articles: " + " | ".join(f"{k} {res[k]:.2f}" for k in ("rouge1", "rouge2", "rougeL", "rougeLsum")))
except (IndexError, KeyError, FileNotFoundError) as e:
    print("CNN/DailyMail test CSV not attached - skipping the ROUGE check", repr(e)[:120])